# 📘 File System Operations (pathlib & os) for Data Engineering
## Databricks Notebook — File Management Mastery

**What you'll learn:**
- pathlib (modern) vs os (legacy) for file operations
- Databricks DBFS and Unity Catalog Volumes
- File-based ingestion patterns (landing → archive → quarantine)
- File format detection, temp files, directory structures
- Storage monitoring and cleanup

**Key rules:**
- Prefer `pathlib` over `os.path` for new code
- In Databricks: DBFS uses `/dbfs/` prefix for Python, `dbfs:/` for dbutils
- Always check existence before reading
- Always use context managers for file handles

**Prerequisites:** Notebooks 01-19

---

---
# 📗 Section 1: File Operations in Data Engineering

**Data lake file flow:**
```
External Source → Landing Zone → Raw/Bronze → Processed/Silver → Archive
                                    ↓ (failures)
                              Quarantine Zone
```

**Databricks file systems:**
- **DBFS:** Legacy distributed file system (being deprecated)
- **Unity Catalog Volumes:** Modern replacement (governed, secure)
- **Local driver disk:** `/tmp/` — temporary only, lost on cluster restart

---
# 📗 Section 2: pathlib (Modern Python)

In [ ]:
from pathlib import Path

# Path objects (cross-platform, readable)
base = Path("/tmp/data_lake")
landing = base / "landing"
raw = base / "raw"
archive = base / "archive"
quarantine = base / "quarantine"

# Create directory structure
for d in [landing, raw, archive, quarantine]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Created data lake structure at: {base}")
print(f"  Landing:    {landing}")
print(f"  Raw:        {raw}")
print(f"  Archive:    {archive}")
print(f"  Quarantine: {quarantine}")

In [ ]:
# Create sample files
from pathlib import Path
import json as json_lib
from datetime import datetime

landing = Path("/tmp/data_lake/landing")

# Drop some sample files
files_created = []
for i in range(1, 6):
    # CSV files
    csv_path = landing / f"orders_{datetime.now().strftime('%Y%m%d')}_{i:03d}.csv"
    csv_path.write_text(f"order_id,amount,status\n{i},100.{i},completed\n{i+10},200.{i},pending")
    files_created.append(csv_path)

# JSON file
json_path = landing / "events_20240615.json"
json_path.write_text(json_lib.dumps([{"event": "click", "user": "u1"}, {"event": "purchase", "user": "u2"}]))
files_created.append(json_path)

# Bad file (for quarantine testing)
bad_path = landing / "corrupted_data.csv"
bad_path.write_text("this is not valid csv\n\x00\x01\x02")
files_created.append(bad_path)

print(f"Created {len(files_created)} files in landing zone")

In [ ]:
# pathlib operations
from pathlib import Path

landing = Path("/tmp/data_lake/landing")

# List all files
print("All files in landing:")
for f in sorted(landing.iterdir()):
    if f.is_file():
        stat = f.stat()
        print(f"  {f.name:<35} {stat.st_size:>6} bytes  {f.suffix}")

# Glob: find specific patterns
print(f"\nCSV files: {[f.name for f in landing.glob('*.csv')]}")
print(f"JSON files: {[f.name for f in landing.glob('*.json')]}")

# File metadata
sample = list(landing.glob("*.csv"))[0]
print(f"\nFile details: {sample.name}")
print(f"  Stem: {sample.stem}")
print(f"  Suffix: {sample.suffix}")
print(f"  Parent: {sample.parent}")
print(f"  Size: {sample.stat().st_size} bytes")
print(f"  Exists: {sample.exists()}")

---
# 📗 Section 3: os and os.path (Legacy)

In [ ]:
import os

# Equivalent operations with os module
landing_path = "/tmp/data_lake/landing"

# List files
files = os.listdir(landing_path)
print(f"Files (os.listdir): {files}")

# Path joining (os.path.join)
full_path = os.path.join(landing_path, "orders_20240615_001.csv")
print(f"\nJoined path: {full_path}")
print(f"Exists: {os.path.exists(full_path)}")
print(f"Is file: {os.path.isfile(full_path)}")
print(f"Size: {os.path.getsize(full_path)} bytes")

# Environment variables (common for configs)
os.environ["PIPELINE_ENV"] = "development"
env = os.environ.get("PIPELINE_ENV", "production")
print(f"\nEnvironment: {env}")

---
# 📗 Section 4: Databricks File System (DBFS)

In [ ]:
# dbutils.fs operations
# Create a directory in DBFS
dbutils.fs.mkdirs("dbfs:/tmp/de_notebook_20/landing")
dbutils.fs.mkdirs("dbfs:/tmp/de_notebook_20/archive")

# Write a file
dbutils.fs.put("dbfs:/tmp/de_notebook_20/landing/test.csv", "id,name,value\n1,Alice,100\n2,Bob,200", overwrite=True)

# List files
files = dbutils.fs.ls("dbfs:/tmp/de_notebook_20/landing/")
print("DBFS files:")
for f in files:
    print(f"  {f.name:<30} {f.size:>8} bytes  isDir={f.isDir()}")

# Read file content (first 100 bytes)
content = dbutils.fs.head("dbfs:/tmp/de_notebook_20/landing/test.csv", 100)
print(f"\nFile content:\n{content}")

In [ ]:
# Copy, move, remove
dbutils.fs.cp("dbfs:/tmp/de_notebook_20/landing/test.csv", "dbfs:/tmp/de_notebook_20/archive/test_archived.csv")
print("Copied to archive")

# List archive
for f in dbutils.fs.ls("dbfs:/tmp/de_notebook_20/archive/"):
    print(f"  Archive: {f.name}")

# Cleanup
dbutils.fs.rm("dbfs:/tmp/de_notebook_20/", recurse=True)
print("\nCleaned up DBFS temp directory")

---
# 📗 Section 5: Unity Catalog Volumes (Modern)

**Volumes** replace DBFS mounts. They're governed by Unity Catalog permissions.

```
/Volumes/<catalog>/<schema>/<volume>/path/to/file.csv
```

| Feature | DBFS Mounts | Unity Catalog Volumes |
|---|---|---|
| Governance | None | Full UC permissions |
| Path format | /mnt/data/ | /Volumes/catalog/schema/vol/ |
| Access control | Cluster-level | User/group-level |
| Recommended | ❌ Legacy | ✅ Modern |

⚠️ Volumes require Unity Catalog (not available on Community Edition).

---
# 📗 Section 6: File-Based Ingestion Patterns

In [ ]:
from pathlib import Path
import shutil
from datetime import datetime

class FileIngestionManager:
    """Manage file-based ingestion with archive and quarantine."""
    
    def __init__(self, base_path):
        self.base = Path(base_path)
        self.landing = self.base / "landing"
        self.archive = self.base / "archive"
        self.quarantine = self.base / "quarantine"
        self.processed_log = []
        
        for d in [self.landing, self.archive, self.quarantine]:
            d.mkdir(parents=True, exist_ok=True)
    
    def detect_new_files(self, pattern="*"):
        """Find unprocessed files in landing zone."""
        processed_names = {r["file"] for r in self.processed_log}
        new_files = [f for f in self.landing.glob(pattern) if f.is_file() and f.name not in processed_names]
        return sorted(new_files, key=lambda f: f.stat().st_mtime)
    
    def validate_file(self, file_path):
        """Basic file validation."""
        if file_path.stat().st_size == 0:
            return False, "Empty file"
        if file_path.suffix not in ('.csv', '.json', '.parquet'):
            return False, f"Unsupported format: {file_path.suffix}"
        # Check for binary/corrupted content in text files
        if file_path.suffix in ('.csv', '.json'):
            try:
                content = file_path.read_text(encoding='utf-8')
                if ' ' in content:
                    return False, "Contains null bytes (corrupted)"
            except UnicodeDecodeError:
                return False, "Not valid UTF-8"
        return True, "OK"
    
    def process_file(self, file_path):
        """Process a single file: validate → ingest → archive/quarantine."""
        is_valid, reason = self.validate_file(file_path)
        
        if is_valid:
            # Archive with timestamp
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            dest = self.archive / f"{ts}_{file_path.name}"
            shutil.move(str(file_path), str(dest))
            self.processed_log.append({"file": file_path.name, "status": "success", "dest": str(dest)})
            return True
        else:
            # Quarantine
            dest = self.quarantine / file_path.name
            shutil.move(str(file_path), str(dest))
            self.processed_log.append({"file": file_path.name, "status": "failed", "reason": reason})
            return False
    
    def run(self, pattern="*"):
        """Process all new files."""
        new_files = self.detect_new_files(pattern)
        success, failed = 0, 0
        
        for f in new_files:
            if self.process_file(f):
                success += 1
            else:
                failed += 1
        
        print(f"Ingestion complete: {success} success, {failed} failed, {len(new_files)} total")
        return self.processed_log

# Run the ingestion manager
manager = FileIngestionManager("/tmp/data_lake")

# Re-create some test files (previous ones were moved)
landing = Path("/tmp/data_lake/landing")
landing.mkdir(parents=True, exist_ok=True)
(landing / "good_data.csv").write_text("id,value\n1,100\n2,200")
(landing / "good_events.json").write_text('[{"event": "test"}]')
(landing / "bad_file.csv").write_text("corrupted\x00data")

results = manager.run("*")
for r in results:
    icon = "✅" if r["status"] == "success" else "❌"
    print(f"  {icon} {r['file']}: {r['status']} {r.get('reason', '')}")

---
# 📗 Section 7: File Metadata & Storage Monitoring

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta

def storage_report(base_path):
    """Generate a storage usage report for a directory tree."""
    base = Path(base_path)
    report = []
    
    for directory in [d for d in base.rglob("*") if d.is_dir()] + [base]:
        files = [f for f in directory.iterdir() if f.is_file()]
        if not files:
            continue
        
        total_size = sum(f.stat().st_size for f in files)
        oldest = min(f.stat().st_mtime for f in files) if files else 0
        newest = max(f.stat().st_mtime for f in files) if files else 0
        
        report.append({
            "directory": str(directory.relative_to(base)),
            "file_count": len(files),
            "total_size_kb": round(total_size / 1024, 2),
            "oldest_file": datetime.fromtimestamp(oldest).strftime("%Y-%m-%d %H:%M") if oldest else "N/A",
            "newest_file": datetime.fromtimestamp(newest).strftime("%Y-%m-%d %H:%M") if newest else "N/A",
        })
    
    print(f"{'='*70}")
    print(f"STORAGE REPORT: {base_path}")
    print(f"{'='*70}")
    print(f"{'Directory':<25} {'Files':<7} {'Size(KB)':<10} {'Oldest':<18} {'Newest'}")
    print("-" * 70)
    for r in sorted(report, key=lambda x: -x["total_size_kb"]):
        print(f"{r['directory']:<25} {r['file_count']:<7} {r['total_size_kb']:<10} {r['oldest_file']:<18} {r['newest_file']}")
    print(f"{'='*70}")

storage_report("/tmp/data_lake")

---
# 🚀 Section 8: Mini Project — Complete File Ingestion System

In [ ]:
# ============================================
# 🎯 YOUR TURN: Build a File Processor
# ============================================
# 1. Create a landing zone at /tmp/my_lake/landing
# 2. Create 5 CSV files and 2 JSON files in it
# 3. Build a processor that:
#    - Detects new files (glob)
#    - Validates format (check extension and content)
#    - Reads valid CSVs into a Spark DataFrame
#    - Moves processed files to /tmp/my_lake/archive
#    - Moves invalid files to /tmp/my_lake/quarantine
# 4. Print a summary report
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pathlib import Path
import shutil
from datetime import datetime

# Setup
base = Path("/tmp/my_lake")
for d in ["landing", "archive", "quarantine"]:
    (base / d).mkdir(parents=True, exist_ok=True)

# Create test files
for i in range(1, 6):
    (base / "landing" / f"data_{i}.csv").write_text(f"id,value\n{i},{i*100}")
(base / "landing" / "events.json").write_text('[{"type": "click"}]')
(base / "landing" / "bad.txt").write_text("unsupported format")

# Process
landing = base / "landing"
processed, failed = 0, 0

for f in sorted(landing.iterdir()):
    if not f.is_file():
        continue
    
    if f.suffix in ('.csv', '.json'):
        # Valid: read and archive
        if f.suffix == '.csv':
            df = spark.read.csv(str(f), header=True, inferSchema=True)
        ts = datetime.now().strftime("%Y%m%d%H%M%S")
        shutil.move(str(f), str(base / "archive" / f"{ts}_{f.name}"))
        processed += 1
    else:
        # Invalid: quarantine
        shutil.move(str(f), str(base / "quarantine" / f.name))
        failed += 1

print(f"Results: {processed} processed, {failed} quarantined")
print(f"Archive: {[f.name for f in (base/'archive').iterdir()]}")
print(f"Quarantine: {[f.name for f in (base/'quarantine').iterdir()]}")

---
# 🏆 Section 9: Interview Questions

## Q1: File-based data ingestion?

1. **Landing zone:** Files arrive here (from SFTP, API downloads, partner drops)
2. **Detect new files:** Compare against processed log (or use Auto Loader)
3. **Validate:** Check format, schema, size, encoding
4. **Process:** Read into Spark, transform, write to Delta
5. **Archive:** Move processed files with timestamp prefix
6. **Quarantine:** Move failed files with error metadata

## Q2: Directory structure for a data lake?

```
/data_lake/
├── landing/          (raw file drops)
├── bronze/           (raw Delta tables)
│   ├── source_a/
│   └── source_b/
├── silver/           (cleaned, validated)
├── gold/             (business aggregates)
├── archive/          (processed source files)
└── quarantine/       (failed files with error logs)
```

## Q3: Handle failed file processing?

1. Move to quarantine directory (don't delete!)
2. Log the error (file name, error type, timestamp)
3. Alert if quarantine grows beyond threshold
4. Periodically review and reprocess or discard
5. Track failure patterns (same source always fails?)

## Q4: pathlib vs os?

**Prefer pathlib:**
- Object-oriented (Path objects, not strings)
- `/` operator for joining (readable)
- Cross-platform by default
- Modern Python (3.4+)

**Use os when:**
- Legacy codebase requires it
- Need os.environ for environment variables
- Low-level operations (os.chmod, os.symlink)

## Q5: Unity Catalog Volumes vs DBFS?

- **Volumes:** Governed by UC permissions, modern, recommended for new projects
- **DBFS:** Legacy, no fine-grained access control, being deprecated
- **Migration:** DBFS mounts → External Volumes → Managed Volumes

## Q6: File format detection with unreliable extensions?

Read first few bytes (magic bytes):
- Parquet: starts with `PAR1`
- Gzip: starts with `\x1f\x8b`
- JSON: starts with `{` or `[`
- CSV: no magic bytes — check for comma-separated patterns

## Q7: Archive/cleanup strategy?

- **Archive:** Move to archive/ with timestamp prefix after successful processing
- **Retention:** Keep archives for 30-90 days (configurable)
- **Cleanup:** Scheduled job deletes archives older than retention period
- **Never delete** quarantined files without review

## Q8: Concurrent file processing?

- Use file-level locking (rename to `.processing` extension while working)
- Or use a control table: claim files by writing your worker_id
- Or use Auto Loader (handles concurrency automatically)
- Write to separate output paths, merge later

---
# 📗 Section 4: Advanced File Operations for DE

Production DE pipelines need robust file handling: atomic writes,
glob patterns, directory watching, and safe cleanup.

In [ ]:
from pathlib import Path
import os
import shutil
import tempfile
import json
import hashlib
import time
from datetime import datetime
from typing import Iterator, List, Optional, Callable

# ============================================================
# PATTERN 1: Atomic File Writes
# ============================================================
# Problem: If a pipeline crashes mid-write, you get a corrupt file.
# Solution: Write to a temp file, then atomically rename.

def atomic_write(path: Path, content: str, encoding: str = "utf-8"):
    """
    Write file atomically: write to temp → rename.
    If the write fails, the original file is untouched.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    
    # Write to temp file in same directory (same filesystem for atomic rename)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    
    try:
        tmp_path.write_text(content, encoding=encoding)
        tmp_path.rename(path)  # Atomic on POSIX, near-atomic on Windows
        return True
    except Exception as e:
        tmp_path.unlink(missing_ok=True)  # Clean up temp file
        raise

def atomic_write_json(path: Path, data: dict, indent: int = 2):
    """Atomically write JSON data."""
    content = json.dumps(data, indent=indent, default=str)
    atomic_write(path, content)

# Test
with tempfile.TemporaryDirectory() as tmpdir:
    output_path = Path(tmpdir) / "config" / "settings.json"
    
    data = {"version": "1.0", "pipeline": "orders_etl", "batch_size": 1000}
    atomic_write_json(output_path, data)
    
    # Verify
    loaded = json.loads(output_path.read_text())
    print(f"Atomic write successful: {loaded}")
    print(f"File exists: {output_path.exists()}")
    print(f"Temp file cleaned up: {not output_path.with_suffix('.json.tmp').exists()}")

In [ ]:
# ============================================================
# PATTERN 2: Glob Patterns for Data Discovery
# ============================================================

def discover_data_files(
    root_dir: Path,
    patterns: List[str] = None,
    recursive: bool = True,
    min_size_bytes: int = 0,
    modified_after: Optional[datetime] = None,
) -> List[dict]:
    """
    Discover data files matching patterns with metadata.
    
    Args:
        root_dir: Root directory to search
        patterns: Glob patterns (e.g., ["*.parquet", "*.csv"])
        recursive: Search subdirectories
        min_size_bytes: Skip files smaller than this
        modified_after: Only include files modified after this time
    """
    root_dir = Path(root_dir)
    patterns = patterns or ["*.parquet", "*.csv", "*.json", "*.avro"]
    
    files = []
    for pattern in patterns:
        glob_fn = root_dir.rglob if recursive else root_dir.glob
        for file_path in glob_fn(pattern):
            if not file_path.is_file():
                continue
            
            stat = file_path.stat()
            
            if stat.st_size < min_size_bytes:
                continue
            
            modified_time = datetime.fromtimestamp(stat.st_mtime)
            if modified_after and modified_time < modified_after:
                continue
            
            files.append({
                "path": str(file_path),
                "name": file_path.name,
                "stem": file_path.stem,
                "suffix": file_path.suffix,
                "size_bytes": stat.st_size,
                "size_kb": round(stat.st_size / 1024, 1),
                "modified_at": modified_time.isoformat(),
                "parent": str(file_path.parent),
                "relative_path": str(file_path.relative_to(root_dir)),
            })
    
    files.sort(key=lambda f: f["modified_at"], reverse=True)
    return files

# Test with temp directory
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    
    # Create sample file structure
    (root / "bronze" / "orders").mkdir(parents=True)
    (root / "bronze" / "customers").mkdir(parents=True)
    (root / "silver").mkdir()
    
    for i in range(3):
        (root / "bronze" / "orders" / f"orders_2024_01_{i+1:02d}.parquet").write_bytes(b"x" * (i+1) * 1000)
        (root / "bronze" / "customers" / f"customers_{i+1}.csv").write_text(f"id,name\n{i},test")
    (root / "silver" / "orders_clean.parquet").write_bytes(b"x" * 5000)
    (root / "README.md").write_text("# Data Lake")
    
    # Discover files
    all_files = discover_data_files(root)
    print(f"All data files: {len(all_files)}")
    for f in all_files:
        print(f"  {f['relative_path']} ({f['size_kb']} KB)")
    
    # Only parquet files
    parquet_files = discover_data_files(root, patterns=["*.parquet"])
    print(f"\nParquet files: {len(parquet_files)}")

In [ ]:
# ============================================================
# PATTERN 3: File Checksum & Deduplication
# ============================================================

def compute_checksum(file_path: Path, algorithm: str = "md5",
                     chunk_size: int = 8192) -> str:
    """Compute file checksum for integrity verification."""
    h = hashlib.new(algorithm)
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

def find_duplicate_files(directory: Path, patterns: List[str] = None) -> dict:
    """Find duplicate files by content hash."""
    patterns = patterns or ["*"]
    hash_map = {}
    
    for pattern in patterns:
        for file_path in directory.rglob(pattern):
            if not file_path.is_file():
                continue
            checksum = compute_checksum(file_path)
            if checksum not in hash_map:
                hash_map[checksum] = []
            hash_map[checksum].append(file_path)
    
    duplicates = {k: v for k, v in hash_map.items() if len(v) > 1}
    return duplicates

# Test
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    
    # Create files (some duplicates)
    content_a = b"order data content A"
    content_b = b"order data content B"
    
    (root / "file1.csv").write_bytes(content_a)
    (root / "file2.csv").write_bytes(content_b)
    (root / "file3.csv").write_bytes(content_a)  # Duplicate of file1
    (root / "subdir").mkdir()
    (root / "subdir" / "file4.csv").write_bytes(content_a)  # Another duplicate
    
    dupes = find_duplicate_files(root, ["*.csv"])
    print(f"Duplicate groups found: {len(dupes)}")
    for checksum, paths in dupes.items():
        print(f"  Hash {checksum[:8]}...: {[p.name for p in paths]}")

## 4.2 Directory Watching & Incremental Processing

In [ ]:
# ============================================================
# PATTERN 4: Incremental File Processing with State
# ============================================================

class IncrementalFileProcessor:
    """
    Process only new/modified files since last run.
    Maintains state in a JSON file.
    """
    
    def __init__(self, state_file: Path):
        self.state_file = Path(state_file)
        self._state = self._load_state()
    
    def _load_state(self) -> dict:
        if self.state_file.exists():
            return json.loads(self.state_file.read_text())
        return {"processed_files": {}, "last_run": None}
    
    def _save_state(self):
        atomic_write_json(self.state_file, self._state)
    
    def get_new_files(self, directory: Path, pattern: str = "*.csv") -> List[Path]:
        """Return files that are new or modified since last run."""
        new_files = []
        
        for file_path in sorted(directory.rglob(pattern)):
            if not file_path.is_file():
                continue
            
            file_key = str(file_path)
            current_mtime = file_path.stat().st_mtime
            current_size = file_path.stat().st_size
            
            prev = self._state["processed_files"].get(file_key)
            
            if prev is None:
                new_files.append(file_path)  # New file
            elif prev["mtime"] != current_mtime or prev["size"] != current_size:
                new_files.append(file_path)  # Modified file
        
        return new_files
    
    def mark_processed(self, file_path: Path, metadata: dict = None):
        """Mark a file as processed."""
        file_key = str(file_path)
        self._state["processed_files"][file_key] = {
            "mtime": file_path.stat().st_mtime,
            "size": file_path.stat().st_size,
            "processed_at": datetime.utcnow().isoformat(),
            **(metadata or {}),
        }
        self._state["last_run"] = datetime.utcnow().isoformat()
        self._save_state()

# Test incremental processing
with tempfile.TemporaryDirectory() as tmpdir:
    data_dir = Path(tmpdir) / "data"
    data_dir.mkdir()
    state_file = Path(tmpdir) / "processor_state.json"
    
    processor = IncrementalFileProcessor(state_file)
    
    # Create initial files
    for i in range(3):
        (data_dir / f"batch_{i:03d}.csv").write_text(f"id,val\n{i},test")
    
    # First run — all files are new
    new_files = processor.get_new_files(data_dir)
    print(f"Run 1 — New files: {[f.name for f in new_files]}")
    for f in new_files:
        processor.mark_processed(f, {"rows": 100})
    
    # Second run — no new files
    new_files = processor.get_new_files(data_dir)
    print(f"Run 2 — New files: {[f.name for f in new_files]}")
    
    # Add a new file
    (data_dir / "batch_003.csv").write_text("id,val\n3,new")
    new_files = processor.get_new_files(data_dir)
    print(f"Run 3 — New files: {[f.name for f in new_files]}")

---
# 📗 Section 5: Practice Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Data Lake Directory Manager
# ============================================================
# Build a utility that manages a medallion architecture directory structure

class DataLakeManager:
    """
    Manages a local data lake directory structure.
    Mirrors the Databricks DBFS/Unity Catalog structure.
    """
    
    def __init__(self, root: Path):
        self.root = Path(root)
        self._setup_structure()
    
    def _setup_structure(self):
        """Create the medallion architecture directories."""
        for layer in ["bronze", "silver", "gold"]:
            (self.root / layer).mkdir(parents=True, exist_ok=True)
        (self.root / "_checkpoints").mkdir(exist_ok=True)
        (self.root / "_logs").mkdir(exist_ok=True)
    
    def get_layer_path(self, layer: str, table: str) -> Path:
        """Get path for a specific table in a layer."""
        path = self.root / layer / table
        path.mkdir(parents=True, exist_ok=True)
        return path
    
    def write_partition(self, layer: str, table: str,
                        partition_key: str, data: str) -> Path:
        """Write data to a partitioned path."""
        table_path = self.get_layer_path(layer, table)
        partition_path = table_path / partition_key
        partition_path.mkdir(parents=True, exist_ok=True)
        
        file_path = partition_path / f"data_{int(time.time())}.json"
        atomic_write(file_path, data)
        return file_path
    
    def list_partitions(self, layer: str, table: str) -> List[str]:
        """List all partitions for a table."""
        table_path = self.root / layer / table
        if not table_path.exists():
            return []
        return sorted([p.name for p in table_path.iterdir() if p.is_dir()])
    
    def get_stats(self) -> dict:
        """Get storage statistics for each layer."""
        stats = {}
        for layer in ["bronze", "silver", "gold"]:
            layer_path = self.root / layer
            files = list(layer_path.rglob("*"))
            data_files = [f for f in files if f.is_file()]
            total_bytes = sum(f.stat().st_size for f in data_files)
            stats[layer] = {
                "file_count": len(data_files),
                "total_kb": round(total_bytes / 1024, 1),
                "tables": [d.name for d in layer_path.iterdir() if d.is_dir()],
            }
        return stats

# Test the data lake manager
with tempfile.TemporaryDirectory() as tmpdir:
    lake = DataLakeManager(Path(tmpdir) / "data_lake")
    
    # Write some data
    for day in ["2024-01-15", "2024-01-16", "2024-01-17"]:
        lake.write_partition("bronze", "orders", f"date={day}",
                             json.dumps({"date": day, "count": 100}))
        lake.write_partition("silver", "orders", f"date={day}",
                             json.dumps({"date": day, "count": 95, "cleaned": True}))
    
    lake.write_partition("gold", "daily_revenue", "2024-01",
                         json.dumps({"month": "2024-01", "revenue": 50000}))
    
    # List partitions
    print("Bronze orders partitions:", lake.list_partitions("bronze", "orders"))
    print("Silver orders partitions:", lake.list_partitions("silver", "orders"))
    
    # Stats
    stats = lake.get_stats()
    print("\nData Lake Stats:")
    for layer, info in stats.items():
        print(f"  {layer}: {info['file_count']} files, {info['total_kb']} KB, tables={info['tables']}")

In [ ]:
# ============================================================
# EXERCISE 2: Log File Archiver
# ============================================================
# Archive old log files: compress and move to archive directory

import gzip

def archive_old_logs(log_dir: Path, archive_dir: Path,
                     days_old: int = 7, dry_run: bool = False) -> dict:
    """
    Archive log files older than N days.
    Compresses with gzip and moves to archive directory.
    """
    log_dir = Path(log_dir)
    archive_dir = Path(archive_dir)
    
    cutoff = time.time() - (days_old * 86400)
    
    archived = []
    skipped = []
    errors = []
    
    for log_file in log_dir.glob("*.log"):
        if log_file.stat().st_mtime > cutoff:
            skipped.append(log_file.name)
            continue
        
        if dry_run:
            archived.append({"file": log_file.name, "action": "would_archive"})
            continue
        
        try:
            archive_dir.mkdir(parents=True, exist_ok=True)
            archive_path = archive_dir / (log_file.name + ".gz")
            
            # Compress
            with open(log_file, "rb") as f_in:
                with gzip.open(archive_path, "wb") as f_out:
                    shutil.copyfileobj(f_in, f_out)
            
            original_size = log_file.stat().st_size
            compressed_size = archive_path.stat().st_size
            ratio = (1 - compressed_size / original_size) * 100 if original_size > 0 else 0
            
            log_file.unlink()  # Remove original
            archived.append({
                "file": log_file.name,
                "original_kb": round(original_size / 1024, 1),
                "compressed_kb": round(compressed_size / 1024, 1),
                "compression_pct": round(ratio, 1),
            })
        except Exception as e:
            errors.append({"file": log_file.name, "error": str(e)})
    
    return {"archived": archived, "skipped": skipped, "errors": errors}

# Test
with tempfile.TemporaryDirectory() as tmpdir:
    log_dir = Path(tmpdir) / "logs"
    archive_dir = Path(tmpdir) / "archive"
    log_dir.mkdir()
    
    # Create old and new log files
    old_log = log_dir / "pipeline_2024_01_01.log"
    old_log.write_text("Log entry 1\n" * 100)
    os.utime(old_log, (time.time() - 10 * 86400,) * 2)  # 10 days old
    
    new_log = log_dir / "pipeline_2024_01_15.log"
    new_log.write_text("Recent log entry\n" * 50)
    
    result = archive_old_logs(log_dir, archive_dir, days_old=7)
    
    print(f"Archived: {len(result['archived'])} files")
    for f in result["archived"]:
        print(f"  {f['file']}: {f['original_kb']} KB → {f['compressed_kb']} KB ({f['compression_pct']}% reduction)")
    print(f"Skipped (recent): {result['skipped']}")
    print(f"Remaining in log_dir: {[f.name for f in log_dir.iterdir()]}")

---
# 📗 Section 5: File Operations in Data Engineering

## Pathlib -- Modern File Handling

Key operations:
- `Path.mkdir(parents=True, exist_ok=True)` -- create directory tree
- `Path.glob('*.json')` -- find files matching pattern
- `Path.rglob('**/*.parquet')` -- recursive search
- `Path.read_text() / write_text()` -- read/write text files
- `Path.stat().st_size` -- file size in bytes
- `Path.rename(dest)` -- move/rename file
- `Path.unlink()` -- delete file
- `Path.exists()` -- check if path exists

## File Processing Patterns for DE

```python
from pathlib import Path
import json

# Process all files for a date
def process_daily_files(base_dir, date_str):
    date_dir = Path(base_dir) / date_str
    if not date_dir.exists():
        return []
    records = []
    for file in sorted(date_dir.glob("*.json")):
        records.extend(json.loads(file.read_text()))
    return records

# Archive processed files
def archive_files(source_dir, archive_dir, pattern="*.json"):
    source = Path(source_dir)
    archive = Path(archive_dir)
    archive.mkdir(parents=True, exist_ok=True)
    for file in source.glob(pattern):
        file.rename(archive / file.name)
```

In [ ]:
# Pathlib patterns for data engineering
from pathlib import Path
import os
import tempfile
import json

print("Pathlib Patterns for Data Engineering")
print("=" * 60)

# Create a temp directory for demo
with tempfile.TemporaryDirectory() as tmpdir:
    base = Path(tmpdir)
    
    # Create directory structure
    (base / "raw" / "orders" / "2024" / "03").mkdir(parents=True)
    (base / "raw" / "orders" / "2024" / "04").mkdir(parents=True)
    (base / "processed").mkdir()
    
    # Create sample files
    for day in range(1, 4):
        file = base / "raw" / "orders" / "2024" / "03" / f"orders_2024-03-{day:02d}.json"
        file.write_text(json.dumps([{"id": day, "amount": day * 100}]))
    
    print("\n1. Directory structure:")
    for p in sorted(base.rglob("*")):
        indent = "  " * (len(p.relative_to(base).parts) - 1)
        icon = "D" if p.is_dir() else "F"
        print(f"  {indent}[{icon}] {p.name}")
    
    print("\n2. Find all JSON files:")
    json_files = sorted(base.rglob("*.json"))
    for f in json_files:
        size = f.stat().st_size
        print(f"  {f.relative_to(base)} ({size} bytes)")
    
    print("\n3. Process files by date pattern:")
    march_files = sorted((base / "raw" / "orders" / "2024" / "03").glob("*.json"))
    all_records = []
    for f in march_files:
        records = json.loads(f.read_text())
        all_records.extend(records)
        print(f"  Processed: {f.name} ({len(records)} records)")
    print(f"  Total: {len(all_records)} records")
    
    print("\n4. File metadata:")
    for f in json_files[:2]:
        stat = f.stat()
        print(f"  {f.name}: {stat.st_size} bytes, stem={f.stem}, suffix={f.suffix}")

print("\n5. Key pathlib operations:")
ops = [
    "Path.mkdir(parents=True, exist_ok=True) -- create directory tree",
    "Path.glob('*.json') -- find files matching pattern",
    "Path.rglob('**/*.parquet') -- recursive search",
    "Path.read_text() / write_text() -- read/write text files",
    "Path.stat().st_size -- file size in bytes",
    "Path.rename(dest) -- move/rename file",
    "Path.unlink() -- delete file",
    "Path.exists() -- check if path exists",
]
for op in ops:
    print(f"  * {op}")


In [ ]:
# ============================================
# 🎯 YOUR TURN: File Processing Pipeline
# ============================================
# Build a function that:
# 1. Scans a directory for all .json files
# 2. Reads each file and counts total records
# 3. Moves processed files to an 'archive' subdirectory
# 4. Returns a summary dict: {filename: record_count}
#
# Use pathlib for all file operations.


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pathlib import Path
import json
import tempfile

def process_and_archive(source_dir):
    source = Path(source_dir)
    archive = source / "archive"
    archive.mkdir(exist_ok=True)
    
    summary = {}
    for file in sorted(source.glob("*.json")):
        records = json.loads(file.read_text())
        summary[file.name] = len(records)
        # Move to archive
        file.rename(archive / file.name)
    
    return summary

# Demo
with tempfile.TemporaryDirectory() as tmpdir:
    # Create test files
    for i in range(3):
        Path(tmpdir, f"batch_{i}.json").write_text(
            json.dumps([{"id": j} for j in range(i * 10, (i+1) * 10)])
        )
    
    print("Before processing:")
    print(f"  Files: {[f.name for f in Path(tmpdir).glob('*.json')]}")
    
    result = process_and_archive(tmpdir)
    
    print("\nAfter processing:")
    print(f"  Summary: {result}")
    print(f"  Archived: {[f.name for f in (Path(tmpdir) / 'archive').glob('*.json')]}")
    print(f"  Remaining in source: {[f.name for f in Path(tmpdir).glob('*.json')]}")


---
# ✅ Notebook Complete!

**What was covered:**
- pathlib: Path objects, glob, file metadata, directory creation
- os module: listdir, walk, environ, path operations
- DBFS: dbutils.fs operations (ls, cp, mv, rm, mkdirs, put, head)
- Unity Catalog Volumes (conceptual — modern replacement)
- File ingestion: FileIngestionManager with validate/archive/quarantine
- Storage monitoring: recursive directory report
- Mini project: complete file processor with Spark integration
- 8 interview questions

**Next:** Notebook 21 — Capstone: E-Commerce Pipeline

---